# 4-5 レポート用 Excel への書き出し

ここまでで揃ったもの:

- 集計結果の DataFrame たち (`by_region` / `by_pref` / `by_product` / `pivot_region_cat` / 明細 `df`)
- グラフ画像 5 枚 (`output/charts/*.png`)

これを **1 つの Excel ファイル** にまとめます。シートは 5 つ + グラフ画像を「概要」シートに貼り付け。**上司に送って違和感のないレポート** が完成します。

## このノートのゴール

`output/report_2026-01.xlsx` を出力する。中身:

| シート | 中身 |
|---|---|
| 概要 | 全社サマリ表 + グラフ画像 (日次推移 / 地域別) |
| 地域別 | 地域ブロック別の売上・利益・件数 |
| 都道府県別 | 47 都道府県のランキング |
| 商品別 | 6 商品の売上・利益・数量 |
| 明細 | 全 47,000 行の元データ |

ポイントは:

- **`pd.ExcelWriter`** で 1 ファイルに複数シートを書く (2-5 の復習)
- **`openpyxl.drawing.image.Image`** で **シートに画像 (PNG) を貼り付け** る (今回の新ネタ)

## Excel との対比

| やりたいこと | Excel (手作業) | pandas + openpyxl |
|---|---|---|
| 複数シートに表を貼る | コピペ × シート数 | `with ExcelWriter: to_excel(sheet_name=…)` |
| 画像を挿入する | 挿入 → 画像 → 場所調整 | `ws.add_image(Image(path), "B10")` |
| 列幅を整える | ダブルクリックで自動調整 | `ws.column_dimensions["A"].width = 14` |
| 来月も同じ体裁で作る | また全部手作業 | スクリプト 1 実行 |

## 準備 — 4-3 までの集計をもう一度組み立て

In [ ]:
# === Colab ===
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = "/content/drive/MyDrive/python-data-basics"

# === ローカル ===
BASE = "."

DATA = f"{BASE}/data/capstone"
SALES_DIR = f"{DATA}/sales_2026-01"
OUT = f"{BASE}/output"
CHARTS = f"{OUT}/charts"

import os
import pandas as pd
from pathlib import Path

os.makedirs(OUT, exist_ok=True)

# --- 4-2 + 4-3 の処理 ---
files = sorted(Path(SALES_DIR).glob("sales_2026-01_*.xlsx"))
dfs = []
for f in files:
    d = pd.read_excel(f, sheet_name="売上")
    d["prefecture_code"] = int(f.stem.split("_")[2])
    dfs.append(d)
all_sales = pd.concat(dfs, ignore_index=True)

master_products = pd.read_excel(f"{DATA}/master_products.xlsx")
master_branches = pd.read_excel(f"{DATA}/master_branches.xlsx")
df = (
    all_sales
    .merge(master_products[["product_code", "category", "cost"]], on="product_code", how="left")
    .merge(master_branches, on="prefecture_code", how="left")
)
df["profit"] = df["amount"] - df["cost"] * df["quantity"]

by_region = df.groupby("region").agg(
    売上=("amount", "sum"), 利益=("profit", "sum"), 件数=("amount", "count"),
).sort_values("売上", ascending=False)
by_pref = df.groupby("prefecture_name").agg(
    売上=("amount", "sum"), 利益=("profit", "sum"), 件数=("amount", "count"),
).sort_values("売上", ascending=False)
by_product = df.groupby(["product_code", "product_name", "category"]).agg(
    売上=("amount", "sum"), 利益=("profit", "sum"), 数量=("quantity", "sum"),
).sort_values("売上", ascending=False)

print(f"集計準備 OK / df = {len(df):,} 行")

## 1. 概要シート用のサマリ表を作る

「**全社で売上いくら、件数いくつ、営業日数いくつ、平均日商いくら**」のような **1 行サマリ** を作ります。

In [ ]:
summary = pd.DataFrame({
    "項目": ["対象月", "営業日数", "取引件数", "売上合計 (円)", "利益合計 (円)", "平均日商 (円)"],
    "値": [
        "2026-01",
        df["date"].nunique(),
        len(df),
        int(df["amount"].sum()),
        int(df["profit"].sum()),
        int(df["amount"].sum() / df["date"].nunique()),
    ],
})
summary

## 2. `ExcelWriter` で複数シートに書き出す

2-5 で習った **`with pd.ExcelWriter(...) as w:`** で、5 つのシートを 1 ファイルに書きます。

In [ ]:
report_path = f"{OUT}/report_2026-01.xlsx"

with pd.ExcelWriter(report_path, engine="openpyxl") as w:
    summary.to_excel(w, sheet_name="概要", index=False)
    by_region.to_excel(w, sheet_name="地域別")           # index は地域名 → 残す
    by_pref.to_excel(w, sheet_name="都道府県別")
    by_product.to_excel(w, sheet_name="商品別")
    df.to_excel(w, sheet_name="明細", index=False)

print(f"書き出し完了 → {report_path}")

**ポイント**:

- `index=False` を付けると、**pandas の行番号を Excel に書かない**
- 逆に `by_region` `by_pref` `by_product` は **`groupby` の結果なので index = 地域名/県名/商品コード**。これは Excel 側でも表示してほしいので **`index=False` を付けない**
- **`engine="openpyxl"`** は、xlsx を書ける標準的なライブラリ。Colab にもプリインストール済み

## 3. 概要シートに「グラフ画像」を貼り込む — `openpyxl`

ここからが今回の **新ネタ**。

今書き出した Excel を **`openpyxl` でもう一度開いて**、概要シートに **PNG を `add_image` で貼り付け** ます。

`openpyxl` は **「pandas で表を書く」よりさらに細かい操作 (画像挿入 / 列幅変更 / セル書式)** ができる、Excel 用のライブラリです。

In [ ]:
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

# さっき書いた Excel を読み込む
wb = load_workbook(report_path)
ws = wb["概要"]

# グラフを貼る — 第 2 引数の "D2" 等で 左上のセル位置 を指定する
ws.add_image(XLImage(f"{CHARTS}/daily_sales.png"),  "D2")
ws.add_image(XLImage(f"{CHARTS}/by_region.png"),    "D24")
ws.add_image(XLImage(f"{CHARTS}/by_pref_top10.png"),"D48")

# 「明細」シートの列幅を少し広げる (デフォルトだと窮屈)
for col, width in zip("ABCDEFGHIJKL", [12, 14, 18, 8, 10, 12, 14, 14, 10, 16, 12, 14]):
    wb["明細"].column_dimensions[col].width = width

# 上書き保存
wb.save(report_path)
print("画像貼り付け + 列幅調整 完了")

**ポイント**:

- `load_workbook(path)` で既存の Excel を **読み込んで Workbook オブジェクトに**
- `wb["シート名"]` で **シートにアクセス**
- `ws.add_image(XLImage("画像パス"), "D2")` で画像を **D2 セル を左上に挿入**
- 最後に **`wb.save(path)`** で書き戻す (同じパスを渡せば上書き)

## 4. 出来たファイルを確認

In [ ]:
size_kb = Path(report_path).stat().st_size / 1024
print(f"出力ファイル: {report_path}")
print(f"サイズ      : {size_kb:.1f} KB")

# シート構成を確認
wb = load_workbook(report_path)
print(f"シート     : {wb.sheetnames}")

Excel で開いて、5 つのシートが揃い、**概要シートに 3 枚のグラフが貼られている** ことを確認してください。

> 💡 **Colab で実行している場合**、左サイドバーのファイルアイコンから `output/report_2026-01.xlsx` を右クリックしてダウンロード → ローカルの Excel で開きます。

## 練習問題

本文のコードを少しだけ変える形の演習です。

1. **概要シートに「商品別 売上」グラフも貼り込む** よう改造してください。本文 cell-11 の `add_image` を **もう 1 行追加** するだけ。位置は `"D72"` あたりに。
2. **明細シートの列幅** を、自分が使いやすい数値に変えて試してください。本文 cell-11 の `[12, 14, 18, ...]` を変更。
3. **「上位 5 県だけ」のシート** `都道府県別_TOP5` を追加してください。本文 cell-8 で **`by_pref.head(5)` を新しいシートに to_excel** するだけ。

In [ ]:
# ここに書いてみてください


<details>
<summary>答えを見る</summary>

```python
# 1. 概要シートに「商品別 売上」グラフを追加
wb = load_workbook(report_path)
ws = wb["概要"]
ws.add_image(XLImage(f"{CHARTS}/by_product.png"), "D72")
wb.save(report_path)

# 2. 列幅は好きな値に
wb = load_workbook(report_path)
for col, width in zip("ABCDEFGHIJKL", [15, 15, 22, 8, 10, 14, 16, 16, 10, 18, 14, 16]):
    wb["明細"].column_dimensions[col].width = width
wb.save(report_path)

# 3. TOP5 だけのシートを追加
with pd.ExcelWriter(report_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
    by_pref.head(5).to_excel(w, sheet_name="都道府県別_TOP5")
```

**3 のコツ**: 既に存在するファイルに **追記** するときは `mode="a"` (append) を付ける。同名のシートがあれば置き換える指定が `if_sheet_exists="replace"`。

</details>

## よくあるエラー

### 1. `PermissionError: [Errno 13] Permission denied`
→ **書き出し先の Excel を Excel アプリで開いたまま** にしている。Excel を閉じてから再実行。

### 2. `add_image` した画像が **小さすぎる / 大きすぎる**
→ PNG の dpi で決まる。画像生成時の `dpi=150` を `dpi=120` などに下げると、貼り付け時のサイズが小さくなる。または `XLImage` 後に `img.width = 600; img.height = 400` のように手動指定。

### 3. `mode="a"` で書き出すと「ファイルが既に存在する」と怒られる
→ pandas のバージョンによっては `if_sheet_exists` パラメータが必須。`mode="a", if_sheet_exists="replace"` を付ける。

### 4. 明細シートが **47,000 行で重い**
→ 実運用では明細を別ファイルに分けることも検討。今回は学習のため 1 ファイルにまとめている。

### 5. グラフが概要シートに **被って** 表示される
→ `add_image(..., セル位置)` のセル位置を **十分離して** 指定する。`D2` `D24` `D48` のように 20 行間隔がちょうどよい。

## まとめ

- 表を複数シートに書くのは **`pd.ExcelWriter`** (2-5 の復習)
- グラフ画像を Excel に貼るのは **`openpyxl.drawing.image.Image` + `ws.add_image`**
- 列幅調整は **`ws.column_dimensions["A"].width = 14`**
- 出来上がるのは **シート 5 つ + グラフ 3 枚入りの月次レポート**。Excel で開けば、上司にそのまま送れる体裁

ここまでで **「47 ファイル読み込み → 結合 → 集計 → グラフ → Excel 書き出し」が一通り出来ました**。

次の **4-6** では、この流れ全部を **1 つの関数 `generate_monthly_report(対象月)`** にまとめて、**来月以降は「セル 1 つ実行」で全部終わる** 形に仕上げます。1-5 で習った関数の集大成です。